# Action-conditioned structure — does training on random *causal* actions induce editable latent structure?

**Direction:** `research/directions/action-conditioned-structure.md` · worker notebook (standalone; the master
`00_master_editability.ipynb` is **not** modified).

**Question (enactivist reframe).** A purely passive next-step predictor may never individuate objects as
*manipulable causes*. Does merely **training on random discrete-token actions that causally move the world** induce
causal/editable structure in the latent — structure we can then read out on the **passive** model (actions held at no-op)?

**Design — three GRUs, all measured PASSIVE (no-op):**
1. **Baseline (1)** — passive GRU on clean dataset 4 (`runs/gru/7_dset4_gru_400epochs`).
2. **Action-cond (2)** — GRU with the action token fed into a widened encoder, trained on the nudge-augmented data.
3. **Perturbed-passive (3)** — plain GRU trained on the **same** nudged trajectories but with the token **withheld**.

The **3→2 gap = action-knowledge** (the headline). The **1→3 gap = perturbation-diversity** control (the world just
jitters more). Models 2 & 3 train on byte-identical trajectories (same 90k base seeds as dataset 4 + nudges; same
val split). Everything below is evaluated on dataset-4 held-out (test + edits) in passive mode with the master
§1–§4 suite. The action nudge (~0.7 units) is deliberately **smaller** than a teleport edit, so action-channel
editing at inference is expected to fall short — the payoff is the passive latent structure, not the action channel.

In [ ]:
# [1] Bootstrap: imports, config, dataset-4 eval splits (passive held-out), baseline model 1.
import sys, os, time, json
sys.path.insert(0, "../../../..")
from dataclasses import replace, asdict
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from tqdm.auto import tqdm
from IPython.display import display, Markdown
import h5py
from torch.utils.data import DataLoader

import pim.eval as ev
from pim.extractors import LinearExtractor, MLPExtractor, StateDefinition
from pim.editors import (probe_decomposition, inject_state, fit_state_subspace,
    project_to_subspace, offmanifold_residual, fit_local_subspace, manifold_steer, gradient_steer)
from pim.editors.manifold_steering import _pca_subspace
from pim.eval.controllability import _rollout
from pim.world_models import load_checkpoint, load_dataset, make_test_loader
from pim.world_models.dataloader import ObservationDataset, build_dataloaders
from pim.world_models.gru import GRUModel, ModelConfig
from pim.world_models.action_gru import ActionGRUModel, ActionModelConfig
from pim.simulator.config import SimConfig
from pim.simulator.actions import generate_action_dataset, n_tokens, token_meaning

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE, NUM_WORKERS, N_OBJ = 512, 6, 2
REPO = "../../../.."
DATA_DIR = f"{REPO}/datasets/4_fixed_refl_inview"       # EVAL (clean, passive, held-out)
ACTION_DIR = f"{REPO}/datasets/5_action_augmented"      # TRAIN (nudge-augmented; base seeds match dataset 4)
OUT = "/tmp/action_conditioned"; os.makedirs(OUT, exist_ok=True)
N_TEST_EVAL = 4000     # test subset for §1-§3 banks (identical subset for every model; bounds probe-fit time)

bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
DT = float(test.config["dataset"]["sim"]["dt"])
NE = min(N_TEST_EVAL, test.n_samples)
sub_loader = DataLoader(ObservationDataset(test.h5_path, np.arange(NE), keys=("obs_intensity",)),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

gru1, info1 = load_checkpoint(f"{REPO}/runs/gru/7_dset4_gru_400epochs/best_model.pt", device=DEVICE)
H = gru1.hidden_size
print(f"device={DEVICE} | eval = dataset4 test[:{NE}] + edits({edits.n_samples}) | H={H}")
print(f"model 1 baseline: {info1.run_name} (ep {info1.epoch}, val {info1.val_loss:.5f})")
print(f"actions: {n_tokens(N_OBJ)} tokens -> {[token_meaning(t) for t in range(n_tokens(N_OBJ))]}")


In [ ]:
# [2] Physical (pos, vel) targets on the eval subset, aligned to hidden states (positions[:, :-1]).
v_test = h5py.File(test.h5_path, "r")["velocities"][:NE, :, :N_OBJ, :].astype(np.float32)  # (NE,40,2,2)
vel_tf = v_test[:, :-1]                                  # (NE,39,2,2) aligned to states
pos_tf = test.positions[:NE, :-1, :N_OBJ, :]            # (NE,39,2,2)
vis_tf = test.is_visible[:NE, :-1, :N_OBJ].all(axis=2)  # (NE,39) both objects visible
obs_eval = test.obs[:NE]                                 # (NE,40,R) noisy passive obs
posflat_tf = pos_tf.reshape(*pos_tf.shape[:2], N_OBJ * 2)               # [x0,y0,x1,y1]
velflat_tf = vel_tf.reshape(*vel_tf.shape[:2], N_OBJ * 2)               # [vx0,vy0,vx1,vy1]
posvel_tf = np.concatenate([posflat_tf, velflat_tf], -1)               # (NE,39,8)
LATE_T = 15
print(f"eval subset NE={NE} | velocity temporal std {float(v_test.std(axis=1).mean()):.2e} (const-vel sim) "
      f"| visible aligned frames {int(vis_tf.sum())}")


In [ ]:
# [3] Themes + generic probe/g-fit helpers + a text+markdown table printer (reused across all sections).
OK = {"blue": "#0072B2", "orange": "#D55E00", "green": "#009E73", "pink": "#CC79A7", "yellow": "#E69F00",
      "grey": "#999999", "sky": "#56B4E9", "violet": "#8172B3"}
def style_ax(ax): ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25, lw=0.6)
plt.style.use("default")

def _fit_regress(X, Y, kind, hidden=256, n_epochs=100, lr=2e-3, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    Din, Dout = X.shape[1], Y.shape[1]
    Xt = torch.from_numpy(X.astype(np.float32)).to(DEVICE); Yt = torch.from_numpy(Y.astype(np.float32)).to(DEVICE)
    if kind == "linear":
        Xa = torch.cat([Xt, torch.ones(Xt.shape[0], 1, device=DEVICE)], 1)
        sol = torch.linalg.lstsq(Xa, Yt).solution
        with torch.no_grad(): pred = Xa @ sol
    else:
        net = nn.Sequential(nn.Linear(Din, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                            nn.Linear(hidden, Dout)).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=lr); bs = 4096; Nn = Xt.shape[0]
        for ep in range(n_epochs):
            perm = torch.randperm(Nn, device=DEVICE)
            for i in range(0, Nn, bs):
                idx = perm[i:i + bs]
                loss = ((net(Xt[idx]) - Yt[idx]) ** 2).mean()
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): pred = net(Xt)
    resid2 = ((pred - Yt) ** 2).sum(0); tot2 = ((Yt - Yt.mean(0, keepdim=True)) ** 2).sum(0)
    r2_pc = (1 - resid2 / torch.clamp(tot2, min=1e-12)).cpu().numpy()
    r2_all = float(1 - resid2.sum() / tot2.sum())
    resid_frac = float((((pred - Yt) ** 2).sum() / (Yt ** 2).sum()).sqrt())
    return pred.cpu().numpy(), r2_all, r2_pc, resid_frac

def fit_probe(feats_tf, y_tf, mask, kind, **kw):
    X = feats_tf[mask]; Y = y_tf[mask]
    pred, r2, r2pc, rfrac = _fit_regress(X, Y, kind, **kw)
    return dict(r2=r2, r2pc=r2pc, resid_frac=rfrac, n=X.shape[0])

def both_table(data, columns, row_hdr=""):
    """data: {rowname: {key: val}}; columns: [(key, disp, fmt)]. Prints a text table AND displays markdown."""
    w = 24
    def cell(v, f): return "nan" if isinstance(v, float) and np.isnan(v) else f.format(v)
    print(f"{row_hdr:>{w}s}" + "".join(f"{d:>{w}s}" for _, d, _ in columns))
    for rn, vals in data.items():
        print(f"{str(rn):>{w}s}" + "".join(f"{cell(vals[k], f):>{w}s}" for k, _, f in columns))
    lines = ["| " + row_hdr + " | " + " | ".join(d for _, d, _ in columns) + " |",
             "|" + "---|" * (len(columns) + 1)]
    for rn, vals in data.items():
        lines.append("| **" + str(rn) + "** | " + " | ".join(cell(vals[k], f) for k, _, f in columns) + " |")
    display(Markdown("\n".join(lines)))
print("helpers ready")


## Metric definitions (formulas up front)

All probes are fit **in-sample** on the passive dataset-4 test subset (`N_TEST_EVAL` samples, identical subset for
every model). "late-t" = frames t ≥ 15 (converged-filter regime); "early-t" = t < 15. Physical reference = **8 DOF**
(2 objects × (2 pos + 2 vel)). RMSE (not MSE) is used throughout for comparability.

| metric | definition |
|---|---|
| hidden state `h` | GRU last-layer state after teacher-forcing the passive (no-op) obs to frame t; `h ∈ R^256` |
| **§1** PCA hull dim @p | # PCA components of the visited-`h` bank for ≥ p of the variance (linear upper bound on dim) |
| **§1** intrinsic dim (TwoNN) | model-free `d = 1/mean(log(r₂/r₁))`, r₁,r₂ = 1st/2nd-NN distances (Facco 2017) |
| **§1** intrinsic dim (MLE) | Levina–Bickel MLE over k=20 NN, bias-corrected ×(k−2)/(k−1) |
| **§1** tangent rotation | mean principal angle (deg) between local-PCA tangents (k=64 NN, top-8) of a state and its NN; higher = more curved |
| **§2** recoverability R² | `1 − ‖Y − probe(h)‖²/‖Y − Ȳ‖²`, Y ∈ {pos, vel}; linear vs MLP probe |
| **§3** fiber residual | `‖h − g(pos,vel)‖ / ‖h‖`, g linear or MLP; 0 = `h` fully a function of the 8-dim physical state (canonical) |
| **§4** readout RMSE | position RMSE of the linear probe read off the edited state vs the teleport target (state-space accuracy) |
| **§4** GT next-step RMSE | RMSE(model's next generated obs, true post-edit obs) — observation-space accuracy |
| **§4** obs-change (% of swap) | RMSE(edited step-0 obs, unsteered step-0 obs) as % of the **true-state-swap** obs-change (100% = a real teleport's effect) |
| **§4** ghost-ray ratio | mean intensity on pre-edit-only rays ÷ unsteered — < 1 means the object left its old location (good) |
| **§4** leave-out local-PCA resid | manifold residency of the edited state (fraction; vs. the real-state reference) |

**§4 references** (never editors): **GT (sim)** = the simulator's clean post-edit obs; **Unsteered** = rollout from the
un-edited state; **True-state swap** = rollout from the state obtained by teacher-forcing the *actual* post-edit
observations — the ceiling any hidden-state editor could reach. **Editors:** Readout injection, MLP-probe gradient,
Global-PCA projection, PCA geodesic, Decoder-gradient (oracle). Metrics/units mirror the master `00_master_editability`.

In [ ]:
# [4] Nudge-augmented dataset (guard: generated once; base seeds match dataset 4 -> byte-identical BASE trajectories).
#     9 tokens: no-op (dominant) + {+x,-x,+y,-y} per object. A token applies a small persistent 0.7-unit position
#     nudge to that object at that frame (rejected -> no-op). actions[s] drives the transition s -> s+1.
SIM4 = SimConfig(**{k: test.config["dataset"]["sim"][k] for k in SimConfig.__dataclass_fields__})
os.makedirs(ACTION_DIR, exist_ok=True)
for split, n, seed in [("train", 90000, 0), ("val", 10000, 90000)]:
    p = f"{ACTION_DIR}/{split}.h5"
    if os.path.exists(p): 
        print(f"exists: {p}"); continue
    generate_action_dataset(p, SIM4, n_samples=n, base_seed=seed, nudge=0.7, p_action=0.15, n_workers=16)
with h5py.File(f"{ACTION_DIR}/train.h5", "r") as f:
    A = f["actions"][:5000]; meta = json.loads(f.attrs["config_json"])
frac = float((A > 0).mean()); hist = np.bincount(A[A > 0].ravel().astype(int), minlength=n_tokens(N_OBJ))
print(f"nudge frac/frame {frac:.3f} | nudge={meta['nudge']} p_action={meta['p_action']} "
      f"| non-no-op token hist {hist[1:].tolist()} (should be roughly balanced)")


In [ ]:
# [5] Train model 2 (action-conditioned) & model 3 (perturbed-passive control) on the SAME nudged data.
#     In-memory training (~7 min/model, 400 epochs). Guard: skip + load if checkpoint already exists.
#     Models 2 & 3 use the SAME data + SAME val split (seed 0, 0.1) as run 7's baseline -> byte-identical trajectories.
def load_action_train():
    with h5py.File(f"{ACTION_DIR}/train.h5", "r") as f:
        OBS = f["obs_intensity"][:].astype(np.float32); ACT = f["actions"][:].astype(np.int64)
    rng = np.random.default_rng(0); perm = rng.permutation(OBS.shape[0]); nval = int(0.1 * OBS.shape[0])
    return torch.from_numpy(OBS), torch.from_numpy(ACT), torch.as_tensor(perm[nval:]), torch.as_tensor(perm[:nval])
def _epoch_mem(model, idx, OBS_t, ACT_t, opt, use_actions, bs=256):
    training = opt is not None; model.train(training)
    if training: idx = idx[torch.randperm(len(idx))]
    tot, nb = 0.0, 0
    with (torch.enable_grad() if training else torch.no_grad()):
        for i in range(0, len(idx), bs):
            bi = idx[i:i + bs]; obs = OBS_t[bi].to(DEVICE)
            pred = model(obs, actions=ACT_t[bi].to(DEVICE))[0] if use_actions else model(obs)[0]
            loss = F.mse_loss(pred, obs[:, 1:, :])
            if training: opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
    return tot / nb
def train_wm(make_model, use_actions, run_name, data, n_epochs=400, lr=1e-3, wd=1e-4, seed=0):
    run_dir = f"{REPO}/runs/gru/{run_name}"; ckpt = f"{run_dir}/best_model.pt"
    torch.manual_seed(seed); np.random.seed(seed); model = make_model().to(DEVICE); mcfg = asdict(model.cfg)
    if os.path.exists(ckpt):
        sd = torch.load(ckpt, map_location=DEVICE); model.load_state_dict(sd["model_state"])
        model.eval(); [p.requires_grad_(False) for p in model.parameters()]
        print(f"{run_name}: loaded ckpt (ep {sd['epoch']}, val {sd['val_loss']:.5f}, n_params {sum(p.numel() for p in model.parameters())})")
        return model
    os.makedirs(run_dir, exist_ok=True); OBS_t, ACT_t, tr_idx, val_idx = data
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd); best = float("inf"); t0 = time.perf_counter()
    for ep in tqdm(range(1, n_epochs + 1), desc=run_name):
        _epoch_mem(model, tr_idx, OBS_t, ACT_t, opt, use_actions)
        va = _epoch_mem(model, val_idx, OBS_t, ACT_t, None, use_actions)
        if va < best:
            best = va
            torch.save({"epoch": ep, "model_state": model.state_dict(), "model_config": mcfg, "val_loss": va, "use_actions": use_actions}, ckpt)
    json.dump({"run_name": run_name, "use_actions": use_actions, "model_config": mcfg, "best_val": best}, open(f"{run_dir}/config.json", "w"), indent=2)
    model.eval(); [p.requires_grad_(False) for p in model.parameters()]
    print(f"{run_name}: {n_epochs} ep in {(time.perf_counter() - t0) / 60:.1f} min | best val {best:.5f}")
    return model
EPOCHS = 400
_need = not (os.path.exists(f"{REPO}/runs/gru/8_action_cond_gru_400ep/best_model.pt") and
             os.path.exists(f"{REPO}/runs/gru/9_perturbed_passive_gru_400ep/best_model.pt"))
DATA = load_action_train() if _need else None
gru2 = train_wm(lambda: ActionGRUModel(ActionModelConfig(input_dim=test.obs_res, hidden_size=H,
               n_actions=n_tokens(N_OBJ), action_embed_dim=16)), True, "8_action_cond_gru_400ep", DATA, n_epochs=EPOCHS)
gru3 = train_wm(lambda: GRUModel(ModelConfig(input_dim=test.obs_res, hidden_size=H)), False,
                "9_perturbed_passive_gru_400ep", DATA, n_epochs=EPOCHS)
del DATA


## Exposition — what the actions actually are (perceptual magnitude + causal use)

Before the metrics, so the action tokens are legible: **(E1)** a demo trajectory with random action tokens
showing each action's effect in observation + world space; **(E2)** a *change-the-action* sanity check — same
input, flip the token at one step → the model's predicted rollout diverges → confirms the action channel is
**causally used** (bears on the leakage question); **(E3)** a 2D world animation (GIF, exportable). The
action-conditioned model is `gru2`, run in passive (no-op) mode except where a token is fed. Demo obs are
rendered **clean** (`obs_noise_std=0`) for legibility; the model itself was trained on **noisy** obs (0.2).
Nudge magnitude = **0.7 world units**, applied as a persistent per-object position offset.

In [ ]:
# [E1] Demo trajectory with random actions: token -> observation-space + world-space effect.
from dataclasses import replace as _dc_replace
from pim.simulator.actions import simulate_with_actions
from pim.simulator.sim import simulate as _simulate
from pim.simulator.renderer import render_scene as _render

gru2.eval()
DEMO_SEED = 7
cfg_demo = _dc_replace(SIM4, seed=DEMO_SEED, n_frames=40, obs_noise_std=0.0)
scene_a, acts_a = simulate_with_actions(cfg_demo, nudge=0.7, p_action=0.25, action_seed=DEMO_SEED + 123)
_, _, obs_a = _render(scene_a)                                   # (T,R) clean obs of the acted world
act_frames = [(s, int(acts_a[s])) for s in range(len(acts_a)) if acts_a[s] != 0]
print(f"demo scene: {len(act_frames)} accepted actions -> " + "; ".join(f"f{s}:{token_meaning(t)}" for s, t in act_frames))

fig = plt.figure(figsize=(14, 6)); fig.patch.set_facecolor("#0a0a14")
gs = fig.add_gridspec(2, 2, height_ratios=[1.5, 1])
axw = fig.add_subplot(gs[0, :]); axx = fig.add_subplot(gs[1, 0]); axy = fig.add_subplot(gs[1, 1])
axw.imshow(obs_a, aspect="auto", cmap="gray", origin="upper")
axw.set_title("(a) observation waterfall of the acted world (clean); dashed line = action frame", color="w", fontsize=10)
axw.set_xlabel("ray", color="w"); axw.set_ylabel("frame", color="w"); axw.tick_params(colors="w")
for s, t in act_frames:
    obj = (t - 1) // 4
    cc = "#00d3ff" if obj == 0 else "#ff6b6b"
    axw.axhline(s + 0.5, color=cc, ls="--", lw=1.2)
    axw.text(1, s, token_meaning(t), color=cc, fontsize=7, va="center")
T = scene_a.positions.shape[0]
for obj, cc in [(0, "#00d3ff"), (1, "#ff6b6b")]:
    axx.plot(range(T), scene_a.positions[:, obj, 0], color=cc, lw=1.6, label=f"obj{obj}")
    axy.plot(range(T), scene_a.positions[:, obj, 1], color=cc, lw=1.6)
for s, t in act_frames:
    axx.axvline(s, color="w", ls=":", lw=0.8, alpha=0.5); axy.axvline(s, color="w", ls=":", lw=0.8, alpha=0.5)
for ax, lab, ttl in [(axx, "object x", "(b) object x vs frame (steps = action nudges)"),
                     (axy, "object y (depth)", "(c) object y vs frame")]:
    ax.set_facecolor("#0a0a14"); ax.set_xlabel("frame", color="w"); ax.set_ylabel(lab, color="w")
    ax.set_title(ttl, color="w", fontsize=9); ax.tick_params(colors="w")
    for sp in ax.spines: ax.spines[sp].set_color("w")
axx.legend(fontsize=7, labelcolor="w", facecolor="#0a0a14")
fig.suptitle("Exposition Fig E1 — action tokens and their effect (nudge = 0.7 units, persistent; cyan=obj0, coral=obj1)", color="w", y=1.0, fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUT}/expo_actions_obs.png", dpi=130, bbox_inches="tight", facecolor="#0a0a14"); display(fig); plt.close(fig)

In [ ]:
# [E2] Change-the-action sanity: identical input up to t0, then feed no-op vs a nudge token -> predictions diverge.
#      A zero difference would mean the model ignores the action; a nonzero, localized shift = the channel is used.
cfg_p = _dc_replace(SIM4, seed=DEMO_SEED + 5, n_frames=40, obs_noise_std=0.0)
scene_p = _simulate(cfg_p); _, _, obs_p = _render(scene_p)          # clean passive obs (no actions applied)
obs_pt = torch.from_numpy(obs_p).float().to(DEVICE)                 # (T,R)
t0, KA = 18, 12
BRANCH = {"no-op (0)": 0, f"{token_meaning(1)} (1)": 1, f"{token_meaning(5)} (5)": 5}

@torch.no_grad()
def rollout_with_action(obs_seq, t0, a_t0, k):
    z = torch.zeros(1, dtype=torch.long, device=DEVICE)
    state = None
    for t in range(t0):
        _, state = gru2.step(obs_seq[t][None], state, action=z)      # teacher-force no-op up to t0
    pred, state = gru2.step(obs_seq[t0][None], state, action=torch.tensor([a_t0], device=DEVICE))  # action at t0
    outs = [pred]
    for _ in range(k - 1):
        p, state = gru2.step(outs[-1], state, action=z); outs.append(p)  # then free-run no-op
    return torch.cat(outs, 0).cpu().numpy()                          # (k,R)

rolls = {name: rollout_with_action(obs_pt, t0, a, KA) for name, a in BRANCH.items()}
noop = rolls["no-op (0)"]
fig, axes = plt.subplots(1, len(BRANCH) + 1, figsize=(4.1 * (len(BRANCH) + 1), 4)); fig.patch.set_facecolor("#0a0a14")
vmax = max(float(r.max()) for r in rolls.values())
for ax, (name, r) in zip(axes[:len(BRANCH)], rolls.items()):
    ax.imshow(r, aspect="auto", cmap="gray", origin="upper", vmin=0, vmax=vmax)
    ax.set_title(name, color="w", fontsize=9); ax.set_xlabel("ray", color="w"); ax.tick_params(colors="w")
axes[0].set_ylabel(f"rollout step (from t0={t0})", color="w")
diffname = [n for n in BRANCH if n != "no-op (0)"][0]
d = rolls[diffname] - noop; amax = max(float(np.abs(d).max()), 1e-6)
axes[-1].imshow(d, aspect="auto", cmap="RdBu_r", origin="upper", vmin=-amax, vmax=amax)
axes[-1].set_title(f"delta obs: [{diffname}] - [no-op]", color="w", fontsize=9)
axes[-1].set_xlabel("ray", color="w"); axes[-1].tick_params(colors="w")
mads = {n: float(np.abs(rolls[n] - noop).mean()) for n in BRANCH if n != "no-op (0)"}
fig.suptitle("Exposition Fig E2 — change-the-action sanity: flipping the token at t0 shifts the predicted rollout -> the action channel is causally used",
             color="w", y=1.03, fontsize=10)
fig.tight_layout(); fig.savefig(f"{OUT}/expo_change_action.png", dpi=130, bbox_inches="tight", facecolor="#0a0a14"); display(fig); plt.close(fig)
print("change-action mean|delta obs| vs no-op (0 => action ignored): " + " | ".join(f"{n}: {v:.4f}" for n, v in mads.items()))

In [ ]:
# [E3] Anim E3 — 2D world animation of the acted demo scene (frustum + object circles). NUMBERED like a figure,
#      slowed ~3x (fps 3 vs the old 5) and HOLDS on action frames so each token's effect is legible.
from matplotlib.patches import Polygon as _Poly, Circle as _Circ
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image as _IPyImage
yn, yf, xn, xf, rad = SIM4.y_near, SIM4.y_far, SIM4.x_near, SIM4.x_far, SIM4.radius
frustum = [(-xn, yn), (xn, yn), (xf, yf), (-xf, yf)]
act_at = {s: t for s, t in act_frames}
T_demo = int(scene_a.positions.shape[0])
PAUSE = 3                                                     # extra held frames on each action frame (pause to read it)
frame_order = []
for f in range(T_demo):
    frame_order.append(f)
    if f in act_at: frame_order += [f] * PAUSE                # hold on action frames
figG, axG = plt.subplots(figsize=(5.6, 6.1)); figG.patch.set_facecolor("#0a0a14"); axG.set_facecolor("#0a0a14")
figG.suptitle("Anim E3 \u2014 2D world with action tokens", color="w", fontsize=11, y=0.97)
axG.add_patch(_Poly(frustum, closed=True, fill=False, edgecolor="#5555bb", lw=1.5))
axG.set_xlim(-xf * 1.1, xf * 1.1); axG.set_ylim(yn - 1, yf + 1); axG.set_aspect("equal")
axG.tick_params(colors="w"); axG.set_xlabel("x", color="w"); axG.set_ylabel("y (depth)", color="w")
for sp in axG.spines: axG.spines[sp].set_color("w")
circles = [_Circ((0, 0), rad, color=c) for c in ["#00d3ff", "#ff6b6b"]]
for cc in circles: axG.add_patch(cc)
ttl = axG.set_title("", color="w", fontsize=10)

def _upd(k):
    f = frame_order[k]
    for obj, cc in enumerate(circles):
        cc.center = (float(scene_a.positions[f, obj, 0]), float(scene_a.positions[f, obj, 1]))
    lab = f"frame {f}" + (f"    ACTION: {token_meaning(act_at[f])}" if f in act_at else "")
    ttl.set_text(lab); ttl.set_color("#ffe14d" if f in act_at else "w")
    return circles + [ttl]

gif_path = f"{OUT}/animE3_action_demo.gif"
try:
    anim = FuncAnimation(figG, _upd, frames=len(frame_order), interval=330, blit=False)
    anim.save(gif_path, writer=PillowWriter(fps=3)); plt.close(figG)     # 3 fps (~1.7x slower) + action-frame holds
    print(f"saved Anim E3 -> {gif_path}  ({len(frame_order)} frames @ 3 fps; PAUSE={PAUSE} on {len(act_at)} action frames)")
    display(_IPyImage(filename=gif_path))
except Exception as e:
    print(f"GIF writer unavailable ({e}); showing static frames (Fig E3) instead.")
    plt.close(figG)
    fr = [0, 10, 20, 30, T_demo - 1]
    figS, axs = plt.subplots(1, len(fr), figsize=(3 * len(fr), 3)); figS.patch.set_facecolor("#0a0a14")
    for ax, f in zip(axs, fr):
        ax.set_facecolor("#0a0a14"); ax.add_patch(_Poly(frustum, closed=True, fill=False, edgecolor="#5555bb", lw=1.2))
        for obj, c in enumerate(["#00d3ff", "#ff6b6b"]):
            ax.add_patch(_Circ((float(scene_a.positions[f, obj, 0]), float(scene_a.positions[f, obj, 1])), rad, color=c))
        ax.set_xlim(-xf * 1.1, xf * 1.1); ax.set_ylim(yn - 1, yf + 1); ax.set_aspect("equal")
        ax.set_title(f"frame {f}" + ("  *action*" if f in act_at else ""), color="w", fontsize=8); ax.tick_params(colors="w")
    figS.suptitle("Fig E3 \u2014 2D world with action tokens (static frames)", color="w", fontsize=10)
    figS.tight_layout(); figS.savefig(f"{OUT}/figE3_action_demo_frames.png", dpi=120, bbox_inches="tight", facecolor="#0a0a14"); display(figS); plt.close(figS)

In [ ]:
# [6] Passive (no-op) teacher-force ALL THREE on the dataset-4 test subset -> hidden-state banks (identical eval data).
MODEL_ORDER = ["Baseline (1)", "Perturbed-passive (3)", "Action-cond (2)"]
MCOLOR = {"Baseline (1)": OK["grey"], "Perturbed-passive (3)": OK["blue"], "Action-cond (2)": OK["green"]}
_obj = {"Baseline (1)": gru1, "Perturbed-passive (3)": gru3, "Action-cond (2)": gru2}
MODELS = {}
for name in MODEL_ORDER:
    m = _obj[name]
    preds, states = ev.teacher_force(m, sub_loader, device=DEVICE)   # passive: observe_sequence -> no-op token
    MODELS[name] = dict(model=m, states=states, color=MCOLOR[name])
    print(f"{name:22s} states {states.shape} | passive next-step MSE "
          f"{float(((preds - obs_eval[:, 1:, :]) ** 2).mean()):.5f}")


## §1 — Geometry of the passive state manifold

Intrinsic dimension (TwoNN, MLE), linear-hull dimension, and local curvature (tangent rotation between neighbours)
of the visited-`h` bank, computed in **passive (no-op)** mode for each model. Physical reference = 8 DOF.

In [ ]:
# [7] §1 Geometry — PCA scree + model-free intrinsic dim (TwoNN, MLE) + tangent-rotation curvature, per model.
def scree(states):
    bank = torch.from_numpy(states.reshape(-1, H)).float().to(DEVICE)
    cum = np.cumsum(_pca_subspace(bank, n_components=H, var_threshold=1.0).explained_variance_ratio.cpu().numpy())
    return cum, {p: int((cum < p).sum()) + 1 for p in (0.70, 0.90, 0.95)}
@torch.no_grad()
def _knn(Q, X, k, chunk=2000):
    out = []
    for i in range(0, Q.shape[0], chunk):
        out.append(torch.topk(torch.cdist(Q[i:i + chunk], X), k + 1, largest=False, dim=1).values)
    return torch.cat(out, 0)
@torch.no_grad()
def two_nn_id(X, sample=20000, seed=0):
    g = torch.Generator(device=X.device).manual_seed(seed)
    idx = torch.randperm(X.shape[0], generator=g, device=X.device)[:min(sample, X.shape[0])]
    v = _knn(X[idx], X, 2); r1, r2 = v[:, 1], v[:, 2]; keep = (r1 > 1e-9) & (r2 > r1)
    return float(1.0 / torch.log(r2[keep] / r1[keep]).mean())
@torch.no_grad()
def mle_id(X, k=20, sample=20000, seed=0):
    g = torch.Generator(device=X.device).manual_seed(seed)
    idx = torch.randperm(X.shape[0], generator=g, device=X.device)[:min(sample, X.shape[0])]
    v = _knn(X[idx], X, k); logT = torch.log(v[:, 1:k + 1].clamp_min(1e-9))
    mk = 1.0 / ((logT[:, k - 1:k] - logT[:, :k - 1]).mean(1)).clamp_min(1e-9)
    return float(mk.mean() * (k - 2) / (k - 1))
@torch.no_grad()
def tangent_curv(X, n_anchors=200, k=64, n_tan=8, seed=0):
    g = torch.Generator(device=X.device).manual_seed(seed)
    aidx = torch.randperm(X.shape[0], generator=g, device=X.device)[:n_anchors]
    def tan(c):
        idx = torch.topk(torch.cdist(X[c:c + 1], X)[0], k + 1, largest=False).indices[1:]
        P = X[idx] - X[idx].mean(0, keepdim=True)
        return torch.linalg.svd(P, full_matrices=False)[2][:n_tan]
    angs = []
    for i in aidx:
        nn1 = int(torch.topk(torch.cdist(X[i:i + 1], X)[0], 2, largest=False).indices[1])
        s = torch.linalg.svdvals(tan(int(i)) @ tan(nn1).T).clamp(-1, 1)
        angs.append(float(torch.rad2deg(torch.arccos(s)).mean()))
    return float(np.mean(angs))
PHYS_DOF = 8; GEO = {}
for name in MODEL_ORDER:
    st = MODELS[name]["states"]; bank = torch.from_numpy(st.reshape(-1, H)).float().to(DEVICE)
    cum, dims = scree(st)
    GEO[name] = dict(cum=cum, hull90=dims[0.90], hull95=dims[0.95],
                     twonn=two_nn_id(bank), mle=mle_id(bank), curv=tangent_curv(bank))
    del bank
    if DEVICE == "cuda": torch.cuda.empty_cache()
print("=== §1 GEOMETRY (passive no-op banks; physical DOF = 8) ===")
both_table({n: GEO[n] for n in MODEL_ORDER},
    [("hull90", "PCA hull @90%", "{:d}"), ("hull95", "PCA hull @95%", "{:d}"),
     ("twonn", "intrinsic TwoNN", "{:.2f}"), ("mle", "intrinsic MLE", "{:.2f}"),
     ("curv", "tangent rot (deg)", "{:.1f}")], row_hdr="model (passive)")


In [ ]:
# [8] Fig 1 — geometry (all three passive models side by side): (a) PCA scree, (b) intrinsic dim vs hull, (c) curvature.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
ax = axes[0]
for n in MODEL_ORDER:
    ax.plot(np.arange(1, len(GEO[n]["cum"]) + 1), GEO[n]["cum"], color=MCOLOR[n], lw=2, label=n)
    ax.axvline(GEO[n]["hull90"], color=MCOLOR[n], ls="--", lw=1)
ax.axhline(0.90, color="0.6", ls=":", lw=1); ax.set_xlim(0, 80); ax.set_xlabel("# PCA components")
ax.set_ylabel("cumulative variance"); ax.set_title("(a) PCA scree (dashed = hull@90%)"); ax.legend(fontsize=8); style_ax(ax)
ax = axes[1]
cats = ["TwoNN", "MLE", "hull@90%"]; x = np.arange(3); w = 0.8 / len(MODEL_ORDER)
for j, n in enumerate(MODEL_ORDER):
    vals = [GEO[n]["twonn"], GEO[n]["mle"], GEO[n]["hull90"]]; off = (j - (len(MODEL_ORDER) - 1) / 2) * w
    ax.bar(x + off, vals, w, color=MCOLOR[n], label=n)
    for xi, v in zip(x + off, vals): ax.text(xi, v + 0.5, f"{v:.1f}", ha="center", fontsize=7)
ax.axhline(PHYS_DOF, color="k", ls="--", lw=1.4, label=f"physical {PHYS_DOF} DOF")
ax.set_xticks(x); ax.set_xticklabels(cats); ax.set_ylabel("dimension")
ax.set_title("(b) intrinsic dim vs linear hull@90%"); ax.legend(fontsize=7); style_ax(ax)
ax = axes[2]
cv = [GEO[n]["curv"] for n in MODEL_ORDER]
ax.bar(range(len(MODEL_ORDER)), cv, color=[MCOLOR[n] for n in MODEL_ORDER])
for xi, v in enumerate(cv): ax.text(xi, v + 0.5, f"{v:.0f}°", ha="center", fontsize=8)
ax.set_xticks(range(len(MODEL_ORDER))); ax.set_xticklabels([n.split()[0] for n in MODEL_ORDER], fontsize=8)
ax.set_ylabel("tangent rotation @NN (deg)"); ax.set_title("(c) local curvature"); style_ax(ax)
fig.suptitle("Fig 1 — Passive state-manifold geometry (Baseline / Perturbed-passive / Action-cond)", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_geometry.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


## §2 — Recoverability of (pos, vel) from the passive hidden state

Linear and MLP probes read the physical state out of a single passive `h`, split into early-t (t<15) and late-t
(t≥15). If action-training helped individuate objects, the passive readout should be *more* recoverable
(higher R²), especially for velocity, on the action-cond model.

In [ ]:
# [9] §2 Recoverability — linear + MLP probes for (pos, vel); velocity early-t vs late-t; per model.
VCOMP = ["vx0", "vy0", "vx1", "vy1"]
def build_feats(states, regime):
    sf = states[:, 1:, :]; win = np.concatenate([states[:, :-1, :], states[:, 1:, :]], -1)
    y = velflat_tf[:, 1:, :]; mask = vis_tf[:, 1:] & vis_tf[:, :-1]; rm = np.zeros_like(mask)
    if regime == "early": rm[:, :LATE_T - 1] = True
    else: rm[:, LATE_T - 1:] = True
    return sf, win, y, (mask & rm)
def run_vel(states, regime):
    sf, win, y, mask = build_feats(states, regime)
    return {("sf", "lin"): fit_probe(sf, y, mask, "linear"), ("sf", "mlp"): fit_probe(sf, y, mask, "mlp"),
            ("win", "mlp"): fit_probe(win, y, mask, "mlp")}
vel_res, pos_res = {}, {}
for name in MODEL_ORDER:
    st = MODELS[name]["states"]
    for reg in ["early", "late"]: vel_res[(name, reg)] = run_vel(st, reg)
    pos_res[(name, "lin")] = fit_probe(st, posflat_tf, vis_tf, "linear")
    pos_res[(name, "mlp")] = fit_probe(st, posflat_tf, vis_tf, "mlp")
print("=== §2 RECOVERABILITY R² (in-sample; higher is better) ===")
rec = {}
for name in MODEL_ORDER:
    o = vel_res[(name, "late")]; oe = vel_res[(name, "early")]
    rec[name] = dict(pos_lin=pos_res[(name, "lin")]["r2"], pos_mlp=pos_res[(name, "mlp")]["r2"],
                     vel_lin=o[("sf", "lin")]["r2"], vel_mlp=o[("sf", "mlp")]["r2"],
                     vel_mlp_e=oe[("sf", "mlp")]["r2"], gain=o[("win", "mlp")]["r2"] - o[("sf", "mlp")]["r2"])
both_table(rec, [("pos_lin", "pos R² lin", "{:.3f}"), ("pos_mlp", "pos R² MLP", "{:.3f}"),
    ("vel_lin", "vel R² lin (late)", "{:.3f}"), ("vel_mlp", "vel R² MLP (late)", "{:.3f}"),
    ("vel_mlp_e", "vel R² MLP (early)", "{:.3f}"), ("gain", "2f-1f MLP (late)", "{:+.3f}")], row_hdr="model (passive)")


In [ ]:
# [10] Fig 2 — recoverability: (a) position lin vs MLP, (b) velocity MLP early vs late, (c) per-component velocity (late MLP).
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.4)); x = np.arange(len(MODEL_ORDER)); w = 0.38
ax = axes[0]
ax.bar(x - w / 2, [pos_res[(n, "lin")]["r2"] for n in MODEL_ORDER], w, color=OK["sky"], label="linear")
ax.bar(x + w / 2, [pos_res[(n, "mlp")]["r2"] for n in MODEL_ORDER], w, color=OK["orange"], label="MLP")
ax.set_xticks(x); ax.set_xticklabels([n.split()[0] for n in MODEL_ORDER], fontsize=8); ax.set_ylim(0, 1.02)
ax.set_ylabel("position R²"); ax.set_title("(a) position readout"); ax.legend(fontsize=8); style_ax(ax)
ax = axes[1]
ax.bar(x - w / 2, [vel_res[(n, "early")][("sf", "mlp")]["r2"] for n in MODEL_ORDER], w, color="0.6", label="early-t")
ax.bar(x + w / 2, [vel_res[(n, "late")][("sf", "mlp")]["r2"] for n in MODEL_ORDER], w, color=OK["green"], label="late-t")
ax.set_xticks(x); ax.set_xticklabels([n.split()[0] for n in MODEL_ORDER], fontsize=8); ax.set_ylim(0, 1.02)
ax.set_ylabel("velocity R² (single-frame MLP)"); ax.set_title("(b) velocity readout"); ax.legend(fontsize=8); style_ax(ax)
ax = axes[2]; xc = np.arange(4); w2 = 0.8 / len(MODEL_ORDER)
for j, n in enumerate(MODEL_ORDER):
    ax.bar(xc + (j - (len(MODEL_ORDER) - 1) / 2) * w2, vel_res[(n, "late")][("sf", "mlp")]["r2pc"], w2, color=MCOLOR[n], label=n)
ax.set_xticks(xc); ax.set_xticklabels(VCOMP); ax.set_ylim(0, 1.02)
ax.set_ylabel("velocity R² per component"); ax.set_title("(c) per-component velocity (late MLP)"); ax.legend(fontsize=7); style_ax(ax)
fig.suptitle("Fig 2 — Recoverability of (pos, vel) from a single passive hidden state", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_recoverability.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


## §3 — Fiber collapse / canonicality

Is the hidden state a function of the 8-dim physical `(pos, vel)`? Fit `g(pos,vel)→h` (linear and MLP) and report
the residual fraction `‖h − g‖/‖h‖`. A lower residual (and a smaller linear→MLP drop) means the passive state is
closer to a canonical function of the physical cause — the kind of structure editing needs.

In [ ]:
# [11] §3 Fiber collapse — residual of the best g(pos,vel)->h (linear & MLP); lower = more canonical.
def fit_g(block, kind):
    _, r2, _, rfrac = _fit_regress(posvel_tf[vis_tf], block[vis_tf], kind, hidden=512, n_epochs=120, lr=1.5e-3)
    return rfrac, r2
fiber = {}
for name in MODEL_ORDER:
    st = MODELS[name]["states"]; lrf, lr2 = fit_g(st, "linear"); mrf, mr2 = fit_g(st, "mlp")
    fiber[name] = dict(lin_rf=lrf, mlp_rf=mrf, mlp_r2=mr2, lin_drop=lrf - mrf)
print("=== §3 FIBER RESIDUAL ||h - g(pos,vel)|| / ||h||  (0 = fully canonical / a function of the 8-dim state) ===")
both_table({n: fiber[n] for n in MODEL_ORDER},
    [("lin_rf", "linear g resid", "{:.3f}"), ("mlp_rf", "MLP g resid", "{:.3f}"),
     ("lin_drop", "linear->MLP drop", "{:+.3f}"), ("mlp_r2", "MLP g R² on h", "{:.3f}")], row_hdr="model (passive)")


In [ ]:
# [12] Fig 3 — fiber residual: (a) linear vs MLP g per model, (b) MLP-g R² on h.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2)); x = np.arange(len(MODEL_ORDER)); w = 0.38
ax = axes[0]
ax.bar(x - w / 2, [fiber[n]["lin_rf"] for n in MODEL_ORDER], w, color=OK["sky"], label="linear g")
ax.bar(x + w / 2, [fiber[n]["mlp_rf"] for n in MODEL_ORDER], w, color=OK["orange"], label="MLP g")
for xi, n in zip(x - w / 2, MODEL_ORDER): ax.text(xi, fiber[n]["lin_rf"] + 0.01, f"{fiber[n]['lin_rf']:.2f}", ha="center", fontsize=7)
for xi, n in zip(x + w / 2, MODEL_ORDER): ax.text(xi, fiber[n]["mlp_rf"] + 0.01, f"{fiber[n]['mlp_rf']:.2f}", ha="center", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels([n.split()[0] for n in MODEL_ORDER], fontsize=8)
ax.set_ylabel("residual fraction ||h - g|| / ||h||"); ax.set_ylim(0, 1.05)
ax.set_title("(a) is h a function of (pos, vel)?  lower = more canonical"); ax.legend(fontsize=8); style_ax(ax)
ax = axes[1]
ax.bar(x, [fiber[n]["mlp_r2"] for n in MODEL_ORDER], color=[MCOLOR[n] for n in MODEL_ORDER])
for xi, n in zip(x, MODEL_ORDER): ax.text(xi, fiber[n]["mlp_r2"] + 0.01, f"{fiber[n]['mlp_r2']:.2f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([n.split()[0] for n in MODEL_ORDER], fontsize=8); ax.set_ylim(0, 1.0)
ax.set_ylabel("MLP g: R² on h"); ax.set_title("(b) variance of h explained by g(pos, vel)"); style_ax(ax)
fig.suptitle("Fig 3 — Fiber collapse / canonicality of the passive hidden state", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_fiber.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


## §4 — Latent editability head-to-head (the headline)

Run the master editor line-up (Readout injection / MLP-probe gradient / Global-PCA projection / PCA geodesic /
Decoder-gradient oracle) on each **passive** model, targeting the dataset-4 teleport edits, with the
GT(sim) / Unsteered / True-state-swap references and obs-space selectivity / ghost / persistence metrics.
If action-training induced editable structure, **Action-cond (2)** should be more steerable (larger obs-change
toward the target, lower readout RMSE, better ghost suppression) than **Perturbed-passive (3)**; **1→3** controls
for perturbation diversity alone.

In [ ]:
# [13] §4 shared edit-set setup (model-independent): teleport targets, sim static renders, ghost/target rays, GT trajectory.
from pim.simulator.sim import Scene
from pim.simulator.renderer import render_scene
SUBSPACE_VAR, LOCAL_VAR, LOCAL_BANK_SIZE = 0.90, 0.90, 50_000
LOCAL_K_GEO = 64; N_EDIT, N_ROLLOUT = 64, 15; K_GEO_ITERS = 120; N_CTX = 6
N = min(N_EDIT, edits.n_samples); ef = edits.edit_frame
tgt_pos_flat = edits.positions[:N, ef, :N_OBJ, :].reshape(N, N_OBJ * 2).astype(np.float32)
vel_edits = h5py.File(edits.h5_path, "r")["velocities"][:N, ef, :N_OBJ, :].astype(np.float32)
tgt = torch.from_numpy(tgt_pos_flat).float().to(DEVICE)
tgt_pv = torch.from_numpy(np.concatenate([tgt_pos_flat, vel_edits.reshape(N, N_OBJ * 2)], 1)).float().to(DEVICE)
gt_traj_obs = edits.clean_obs[:N, ef:ef + N_ROLLOUT, :].astype(np.float32)     # true post-edit clean obs (never a model)
ctx_obs = edits.obs[:N, ef - N_CTX:ef, :].astype(np.float32)             # shared pre-edit context
gt_obs_edit_frame = torch.from_numpy(edits.clean_obs[:N, ef, :]).float().to(DEVICE)
OBS_RES = gt_traj_obs.shape[-1]
sim = test.config["dataset"]["sim"]
cfg1 = SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                 n_objects=N_OBJ, radius=sim["radius"], n_frames=1, dt=sim["dt"], obs_res=sim["obs_res"],
                 refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                 obs_noise_std=0.0, boundary="open", always_in_frustum=False)
REFL = np.array([sim["refl_min"], sim["refl_max"]], np.float32); RAD = np.array([sim["radius"]] * N_OBJ, np.float32)
COLc = np.tile(np.array([[1, 1, 1]], np.float32), (N_OBJ, 1))
tgt_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32); pre_pos = edits.positions[:N, ef - 1, :N_OBJ, :].astype(np.float32)
tgt_render_id = np.zeros((N, OBS_RES), np.int64); tgt_render_int = np.zeros((N, OBS_RES), np.float32); pre_render_id = np.zeros((N, OBS_RES), np.int64)
for i in range(N):
    sc = Scene(positions=tgt_pos[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32), radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, rid, rint = render_scene(sc); tgt_render_id[i], tgt_render_int[i] = rid[0], rint[0]
    scp = Scene(positions=pre_pos[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32), radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, ridp, _ = render_scene(scp); pre_render_id[i] = ridp[0]
edit_obj = edits.edit_object[:N]
ghost_mask = np.zeros((N, OBS_RES), bool); target_mask = np.zeros((N, OBS_RES), bool)
for i in range(N):
    ghost_mask[i] = (pre_render_id[i] == edit_obj[i]) & (tgt_render_id[i] != edit_obj[i])
    target_mask[i] = (tgt_render_id[i] == edit_obj[i])
def centroid(mask_row):
    idx = np.where(mask_row)[0]; return idx.mean() if idx.size else np.nan
teleport = np.linalg.norm(tgt_pos - pre_pos, axis=-1)[np.arange(N), edit_obj]
has_ghost = ghost_mask.sum(1) >= 3; SAMPLES = list(np.argsort(teleport * has_ghost)[::-1][:3])
print(f"N={N} edit_frame={ef} rollout={N_ROLLOUT} | ghost rays {int(ghost_mask.sum())} | "
      f"waterfall samples {SAMPLES} (teleport {[round(float(teleport[s]), 2) for s in SAMPLES]})")


In [ ]:
# [14] §4 per-model prep: warm-up h0 (pre-edit context), linear pos probe, frozen MLP (pos,vel) probe,
#      global-PCA subspace, local bank, and the true-state-swap state (teacher-force the ACTUAL post-edit obs).
@torch.no_grad()
def tf_hidden_at(model, obs_seqs, frame):
    out = np.zeros((obs_seqs.shape[0], H), np.float32)
    for i in range(obs_seqs.shape[0]):
        ot = torch.from_numpy(obs_seqs[i]).float().to(DEVICE); state = None
        for t in range(frame + 1):
            _, state = model.step(ot[t].unsqueeze(0), state)      # passive no-op
        out[i] = model.flat_state(state).squeeze(0).cpu().numpy()
    return out
def prep_model(model, states, name):
    sdef = StateDefinition(name="positions", state_shape=(N_OBJ, 2), extract_fn=lambda b: b["positions"])
    lin = LinearExtractor(H, sdef, use_lstsq=True); lin.fit(states, pos_tf, mask=vis_tf, device=DEVICE); lin = lin.to(DEVICE).eval()
    A, b_, A_pinv = probe_decomposition(lin)
    sdef_pv = StateDefinition(name="posvel", state_shape=(N_OBJ * 4,), extract_fn=lambda b: b)
    mlp_pv = MLPExtractor(H, sdef_pv, mlp_hidden=128, n_epochs=30, lr=5e-3)
    mlp_pv.fit(states, posvel_tf, mask=vis_tf, device=DEVICE); mlp_pv = mlp_pv.to(DEVICE).eval()
    sub = fit_state_subspace(states, var_threshold=SUBSPACE_VAR)
    sub = replace(sub, mean=sub.mean.to(DEVICE), basis=sub.basis.to(DEVICE),
                  explained_variance_ratio=sub.explained_variance_ratio.to(DEVICE))
    ba = states.reshape(-1, H)
    bidx = np.random.RandomState(0).choice(ba.shape[0], size=min(LOCAL_BANK_SIZE, ba.shape[0]), replace=False)
    bank = torch.from_numpy(ba[bidx]).float().to(DEVICE)
    warm = ev.warm_up_to_edit(model, edits.obs[:N], ef, n_viz=N, n_ctx_show=8, device=DEVICE)
    h0 = torch.from_numpy(warm.h_at_edit[:N]).float().to(DEVICE)
    h_swap = torch.from_numpy(tf_hidden_at(model, edits.obs[:N], ef)).float().to(DEVICE)
    return dict(model=model, A=A, b=b_, A_pinv=A_pinv, mlp_pv=mlp_pv, sub=sub, bank=bank, h0=h0, h_swap=h_swap)
WM = {name: prep_model(MODELS[name]["model"], MODELS[name]["states"], name) for name in MODEL_ORDER}
def readout(P, h): return h @ P["A"].T + P["b"]
def readout_rmse(P, h): return float((readout(P, h) - tgt).pow(2).mean().sqrt())
for n in MODEL_ORDER:
    print(f"{n:22s} un-edited readout RMSE {readout_rmse(WM[n], WM[n]['h0']):.3f} | "
          f"true-state-swap readout RMSE {readout_rmse(WM[n], WM[n]['h_swap']):.3f}")


In [ ]:
# [15] §4 real-state references: leave-out local-PCA residual (excludes the query's own NN) + global-PCA hull residual.
@torch.no_grad()
def loo_local_resid(h_batch, bank, k_neighbors=LOCAL_K_GEO, n_probe=100, var_threshold=LOCAL_VAR):
    hb = h_batch if isinstance(h_batch, torch.Tensor) else torch.as_tensor(h_batch, device=DEVICE, dtype=torch.float32)
    fracs = []
    for i in range(min(n_probe, hb.shape[0])):
        q = hb[i].reshape(-1); d = torch.cdist(q[None], bank)[0]
        idx = torch.topk(d, min(k_neighbors + 1, bank.shape[0]), largest=False).indices[1:]
        sub = _pca_subspace(bank[idx], n_components=None, var_threshold=var_threshold)
        proj = project_to_subspace(q[None], sub)[0]
        fracs.append(float((q - proj).norm()) / max(float((q - sub.mean).norm()), 1e-9))
    return float(np.mean(fracs))
REAL_LOO, REAL_GLOB = {}, {}
for n in MODEL_ORDER:
    P = WM[n]
    REAL_LOO[n] = loo_local_resid(P["bank"][:200], P["bank"], n_probe=200)
    REAL_GLOB[n] = float(offmanifold_residual(P["bank"][:2000], P["sub"]).mean())
    print(f"{n:22s} real-state leave-out local-PCA resid {REAL_LOO[n]:.3f} (frac) | global-PCA hull resid {REAL_GLOB[n]:.3f}")


In [ ]:
# [16] §4 run the five master editors on all three passive models.
ED_ORDER = ["Readout injection", "MLP-probe gradient", "Global-PCA projection", "PCA geodesic", "Decoder gradient"]
@torch.no_grad()
def pca_geodesic(P, h_start, target, k_local=LOCAL_K_GEO, const_step=None, k_iters=K_GEO_ITERS, desc="geo"):
    A, b_, A_pinv, bank = P["A"], P["b"], P["A_pinv"], P["bank"]
    if const_step is None:
        const_step = 0.34 * float((inject_state(h_start, target, A, A_pinv, b_) - h_start).norm(dim=-1).mean())
    Nn = h_start.shape[0]; h_out = torch.empty_like(h_start); rmse_log = np.full((Nn, k_iters + 1), np.nan)
    for i in tqdm(range(Nn), desc=desc, leave=False):
        h = h_start[i:i + 1]; t = target[i:i + 1]; rmse_log[i, 0] = float((h @ A.T + b_ - t).pow(2).mean().sqrt())
        for kk in range(k_iters):
            d = inject_state(h, t, A, A_pinv, b_) - h; nrm = d.norm(); dhat = d / nrm if float(nrm) > 1e-12 else d
            h_step = h + const_step * dhat
            sub = fit_local_subspace(bank, h_step[0], k_neighbors=k_local, var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK_SIZE)
            h = project_to_subspace(h_step, sub); rmse_log[i, kk + 1] = float((h @ A.T + b_ - t).pow(2).mean().sqrt())
        h_out[i] = h[0]
    return h_out, rmse_log, const_step
def decoder_grad_edit(model, h_init, target_obs, n_iter=400, lr=0.05):
    h = h_init.clone().detach().requires_grad_(True); opt = torch.optim.Adam([h], lr=lr)
    with torch.backends.cudnn.flags(enabled=False):
        for _ in range(n_iter):
            loss = ((model.decode(model.state_from_flat(h)) - target_obs) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return h.detach(), float(loss.item())
EDITS4, GEO_LOG, CONST_STEP = {}, {}, {}
for n in MODEL_ORDER:
    P = WM[n]; h0 = P["h0"]; A, b_, A_pinv = P["A"], P["b"], P["A_pinv"]; E = {}
    E["Readout injection"] = inject_state(h0, tgt, A, A_pinv, b_)
    outs = []
    for i in tqdm(range(N), desc=f"MLP-grad {n}", leave=False):
        h_i, _ = gradient_steer(h0[i:i + 1], tgt_pv[i:i + 1], P["mlp_pv"], n_steps=200, lr=0.01); outs.append(h_i)
    E["MLP-probe gradient"] = torch.cat(outs, 0)
    E["Global-PCA projection"] = manifold_steer(h0, tgt, lambda h, t: inject_state(h, t, A, A_pinv, b_), P["sub"], n_iters=50)
    E["PCA geodesic"], GEO_LOG[n], CONST_STEP[n] = pca_geodesic(P, h0, tgt, desc=f"geodesic {n}")
    E["Decoder gradient"], dl = decoder_grad_edit(P["model"], h0, gt_obs_edit_frame)
    EDITS4[n] = E
    print(f"{n:22s} geodesic readout {np.nanmean(GEO_LOG[n][:, 0]):.2f}->{np.nanmean(GEO_LOG[n][:, -1]):.2f} | dec MSE {dl:.5f}")
    print("   readout RMSE: " + " | ".join(f"{k} {readout_rmse(P, h):.2f}" for k, h in E.items()))


In [ ]:
# [17] §4 model rollouts from every reference/editor state (passive free-run; step-0 = decode without advancing).
REF_ORDER = ["Unsteered", "True-state swap"]
COL = {"GT (sim)": "k", "Unsteered": OK["grey"], "True-state swap": OK["sky"],
       "Readout injection": OK["yellow"], "MLP-probe gradient": OK["orange"],
       "Global-PCA projection": OK["green"], "PCA geodesic": OK["blue"], "Decoder gradient": OK["pink"]}
@torch.no_grad()
def rollout_from_flat(model, h_array, n_rollout):
    out = []
    for i in range(h_array.shape[0]):
        h = torch.as_tensor(h_array[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        o, _ = _rollout(model, h, n_rollout); out.append(o)
    return np.stack(out)
ROLL = {}
for n in MODEL_ORDER:
    P = WM[n]; states4 = {"Unsteered": P["h0"], "True-state swap": P["h_swap"], **EDITS4[n]}
    ROLL[n] = {k: rollout_from_flat(P["model"], h.detach().cpu().numpy(), N_ROLLOUT) for k, h in states4.items()}
print("rollouts done:", {n: ROLL[n]["Unsteered"].shape for n in MODEL_ORDER})


In [ ]:
# [18] §4 metric suite (same metrics/units for all three passive models); per-step tables feed Fig 4.
def rms(a, b): return float(np.sqrt(((a - b) ** 2).mean()))
METRICS, STEP_RMSE, SWAP_CHG, DIST_UNS = {}, {}, {}, {}
for n in MODEL_ORDER:
    P = WM[n]; obs_u = ROLL[n]["Unsteered"]
    swap_chg = rms(ROLL[n]["True-state swap"][:, 0, :], obs_u[:, 0, :]); SWAP_CHG[n] = swap_chg
    rows = {}
    for k in REF_ORDER + ED_ORDER:
        o = ROLL[n][k]
        h = {"Unsteered": P["h0"], "True-state swap": P["h_swap"]}.get(k)
        h = EDITS4[n][k] if h is None else h
        chg = rms(o[:, 0, :], obs_u[:, 0, :])
        ghost = (float(o[:, 0, :][ghost_mask].mean() / max(obs_u[:, 0, :][ghost_mask].mean(), 1e-6))
                 if ghost_mask.sum() > 0 else float("nan"))
        rows[k] = dict(readout=readout_rmse(P, h), nextstep=rms(o[:, 1, :], gt_traj_obs[:, 1, :]),
                       obs_chg=chg, pct_swap=100 * chg / max(swap_chg, 1e-9), ghost=ghost,
                       loo_resid=loo_local_resid(h, P["bank"], n_probe=min(64, N)),
                       glob_resid=float(offmanifold_residual(h, P["sub"]).mean()))
    METRICS[n] = rows
    STEP_RMSE[n] = {k: [rms(ROLL[n][k][:, s, :], gt_traj_obs[:, s, :]) for s in range(N_ROLLOUT)] for k in REF_ORDER + ED_ORDER}
    DIST_UNS[n] = {k: [rms(ROLL[n][k][:, s, :], obs_u[:, s, :]) for s in range(N_ROLLOUT)] for k in ["True-state swap"] + ED_ORDER}
M4 = [("readout", "readout RMSE (pos)", "{:.3f}"), ("nextstep", "GT next-step RMSE", "{:.3f}"),
      ("obs_chg", "obs-change", "{:.3f}"), ("pct_swap", "% of swap", "{:.1f}"), ("ghost", "ghost ratio", "{:.3f}"),
      ("loo_resid", "loo local-PCA resid", "{:.3f}")]
for n in MODEL_ORDER:
    print(f"=== {n} — editor metrics (refs: true-state-swap obs-change {SWAP_CHG[n]:.3f} = 100%; real-state loo {REAL_LOO[n]:.2f}) ===")
    both_table(METRICS[n], M4, row_hdr=str(n))


In [ ]:
# [19] Fig 4 — editing head-to-head, one ROW per model: (a) readout vs next-step obs, (b) per-step GT-traj RMSE, (c) manifold residency.
fig, axes = plt.subplots(len(MODEL_ORDER), 3, figsize=(18, 4.3 * len(MODEL_ORDER)))
steps = np.arange(N_ROLLOUT)
panel_tags = [["a", "b", "c"], ["d", "e", "f"], ["g", "h", "i"]]
for r, n in enumerate(MODEL_ORDER):
    Mx = METRICS[n]; pa, pb, pc = axes[r]; tg = panel_tags[r]
    for k in REF_ORDER + ED_ORDER:
        pa.scatter(Mx[k]["readout"], Mx[k]["nextstep"], s=95, color=COL[k],
                   marker="s" if k in REF_ORDER else "o", edgecolor="k", zorder=3)
    pa.set_xlabel("readout RMSE (pos units)"); pa.set_ylabel("GT next-step RMSE (obs)")
    pa.set_title(f"({tg[0]}) {n} — state vs obs accuracy"); style_ax(pa)
    for k in REF_ORDER + ED_ORDER:
        pb.plot(steps, STEP_RMSE[n][k], color=COL[k], lw=1.6, marker="o", ms=3)
    pb.set_xlabel("rollout step (0 = edit frame)"); pb.set_ylabel("RMSE(gen, true post-edit obs)")
    pb.set_title(f"({tg[1]}) {n} — per-step GT-trajectory RMSE"); style_ax(pb)
    names_c = REF_ORDER + ED_ORDER; vals = [Mx[k]["loo_resid"] for k in names_c]
    pc.bar(range(len(names_c)), vals, color=[COL[k] for k in names_c])
    for i, v in enumerate(vals): pc.text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=7)
    pc.axhline(REAL_LOO[n], color="0.3", ls="--", lw=1.3)
    pc.text(len(names_c) - 0.5, REAL_LOO[n] + 0.015, f"real {REAL_LOO[n]:.2f}", ha="right", fontsize=7, color="0.3")
    pc.set_xticks(range(len(names_c))); pc.set_xticklabels(names_c, rotation=25, ha="right", fontsize=7)
    pc.set_ylabel("leave-out local-PCA resid"); pc.set_title(f"({tg[2]}) {n} — manifold residency"); style_ax(pc)
handles = [Line2D([0], [0], marker="s" if k in REF_ORDER else "o", linestyle="none", markersize=8,
                  markerfacecolor=COL[k], markeredgecolor="k", label=k) for k in REF_ORDER + ED_ORDER]
fig.legend(handles=handles, loc="upper center", ncol=7, fontsize=9, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle("Fig 4 — Editing head-to-head (rows: Baseline / Perturbed-passive / Action-cond)", y=1.015, fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.965]); fig.savefig(f"{OUT}/fig4_editor_metrics.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


In [ ]:
# [20] Fig 5 — editor waterfalls (dark), one figure per model: N_CTX observed context (noisy for editors; clean under GT)
#      rows, then the TEACHER-FORCED true edit frame (ef, observed), then the free-run model rollout (ef+1 onward).
#      Showing ef teacher-forced fixes the GRU +1 decode offset so every column aligns to sim frames honestly.
DARK_BG, DARK_TEXT, DARK_TICK, EDIT_LINE, TF_LINE = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850", "#FFD166"
WATERFALL_COLS = ["GT (sim)"] + REF_ORDER + ED_ORDER
tf_ef = edits.clean_obs[:N, ef, :].astype(np.float32)          # true post-edit obs AT ef (teacher-forced / observed)
def editor_waterfall_fig(n, tag, fname):
    nc = len(WATERFALL_COLS)
    fig, axes = plt.subplots(len(SAMPLES), nc, figsize=(3.0 * nc, 3.4 * len(SAMPLES)), squeeze=False, facecolor=DARK_BG)
    for r, smp in enumerate(SAMPLES):
        tcx = centroid(tgt_render_id[smp] == edit_obj[smp]); pcx = centroid(pre_render_id[smp] == edit_obj[smp])
        for c, k in enumerate(WATERFALL_COLS):
            ax = axes[r][c]
            if k == "GT (sim)":
                ctx = edits.clean_obs[smp, ef - N_CTX:ef, :].astype(np.float32)
                roll = edits.clean_obs[smp, ef + 1:ef + N_ROLLOUT, :].astype(np.float32)        # ef+1 .. ef+N_ROLLOUT-1 (free-run ref)
            else:
                ctx = ctx_obs[smp]
                roll = ROLL[n][k][smp][1:]                                                       # free-run ef+1 onward (drop step0=ef; ef is the shared true row)
            panel = np.clip(np.concatenate([ctx, tf_ef[smp][None], roll], 0), 0, 1)             # ctx | ef (observed) | rollout
            ax.set_facecolor(DARK_BG)
            for sp in ax.spines.values(): sp.set_edgecolor(DARK_TICK)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            ax.axhline(N_CTX - 0.5, color=EDIT_LINE, lw=1.2, ls="--", alpha=0.85)                # context -> edit frame
            ax.axhline(N_CTX + 0.5, color=TF_LINE, lw=1.1, ls=":", alpha=0.9)                    # edit frame -> free-run (ef+1)
            if not np.isnan(tcx): ax.axvline(tcx, color="#00E676", lw=1.5, alpha=0.9)
            if not np.isnan(pcx): ax.axvline(pcx, color="#FF5252", ls="--", lw=1.5, alpha=0.9)
            if r == 0: ax.set_title(k, fontsize=9, color=DARK_TEXT)
            if c == 0:
                ax.set_ylabel(f"sample {smp} (tel {teleport[smp]:.1f})\nsim frame", fontsize=8, color=DARK_TEXT)
                ax.set_yticks([0, N_CTX, N_CTX + 5, N_CTX + 10]); ax.set_yticklabels([ef - N_CTX, ef, ef + 5, ef + 10])
            else:
                ax.set_yticks([])
            ax.set_xlabel("ray", fontsize=8, color=DARK_TEXT); ax.tick_params(colors=DARK_TICK, labelsize=7)
    handles = [Line2D([0], [0], color="#00E676", lw=2.2, label="target (post-edit)"),
               Line2D([0], [0], color="#FF5252", ls="--", lw=2.2, label="ghost (pre-edit)"),
               Line2D([0], [0], color=EDIT_LINE, ls="--", lw=2.2, label="edit frame"),
               Line2D([0], [0], color=TF_LINE, ls=":", lw=2.2, label="ef = true post-edit obs (edit target); rows below = free-run from the edited state (ef+1 →)")]
    fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=9.5, frameon=False, labelcolor=DARK_TEXT, bbox_to_anchor=(0.5, 0.995))
    fig.suptitle(f"Fig {tag} — {n} editor waterfalls (ef teacher-forced; rollout = free-run from the edited state, ef+1 onward)", y=1.0, fontsize=12, color=DARK_TEXT)
    fig.tight_layout(rect=[0, 0, 1, 0.95]); fig.savefig(f"{OUT}/{fname}", dpi=130, bbox_inches="tight", facecolor=DARK_BG); display(fig); plt.close(fig)
editor_waterfall_fig("Baseline (1)", "5a", "fig5a_waterfalls_baseline.png")
editor_waterfall_fig("Perturbed-passive (3)", "5b", "fig5b_waterfalls_perturbed.png")
editor_waterfall_fig("Action-cond (2)", "5c", "fig5c_waterfalls_action.png")

In [ ]:
# [21] Completeness (secondary): can the ACTION CHANNEL itself produce an edit? Free-run the action-cond model
#      from the un-edited state while feeding nudge tokens toward each teleport target; measure obs reach.
@torch.no_grad()
def action_rollout(model, h_flat, tokens):
    state = model.state_from_flat(h_flat)
    obs = [model.decode(state).squeeze(0).cpu().numpy()]
    for tok in tokens:
        obs_hat = model.decode(state)
        pred, state = model.step(obs_hat, state, action=torch.tensor([tok], device=DEVICE))
        obs.append(pred.squeeze(0).cpu().numpy())
    return np.stack(obs)

P2 = WM["Action-cond (2)"]
chg_pct, toward = [], []
for i in range(N):
    obj = int(edit_obj[i]); dvec = tgt_pos[i, obj] - pre_pos[i, obj]
    toks = []
    for _ in range(N_ROLLOUT - 1):
        if abs(dvec[0]) >= abs(dvec[1]):
            toks.append(1 + obj * 4 + (0 if dvec[0] > 0 else 1))
        else:
            toks.append(1 + obj * 4 + (2 if dvec[1] > 0 else 3))
    ar = action_rollout(P2["model"], P2["h0"][i:i + 1], toks)          # (N_ROLLOUT, R)
    uns = ROLL["Action-cond (2)"]["Unsteered"][i]                        # (N_ROLLOUT, R)
    f = N_ROLLOUT - 1
    chg_pct.append(100 * rms(ar[f], uns[f]) / max(SWAP_CHG["Action-cond (2)"], 1e-9))
    toward.append(rms(uns[f], tgt_render_int[i]) - rms(ar[f], tgt_render_int[i]))  # >0 = moved toward target
chg_pct, toward = np.array(chg_pct), np.array(toward)
print("=== Action-channel edit (secondary completeness) ===")
print(f"obs-change induced by driving tokens (final step): mean {chg_pct.mean():.1f}% of a true-state swap")
print(f"moved TOWARD the teleport target: mean d(RMSE)={toward.mean():+.3f} | frac samples moved toward = {(toward>0).mean():.2f}")
print("Read: the small persistent nudge shifts the observation but cannot reach a full frustum-spanning teleport in a")
print("free-run (obs-change << 100% swap) — expected per the brief; the payoff is the PASSIVE latent structure above.")


## Summary — does action-training move §1–§4, and does the 3→2 gap localize it to *action-knowledge*?

`Baseline (1)` = clean-data passive GRU · `Perturbed-passive (3)` = same nudged trajectories, token withheld ·
`Action-cond (2)` = same nudged trajectories, token fed. **1→3** isolates perturbation-diversity; **3→2** isolates
action-knowledge (the enactivist prediction). All numbers are on the passive (no-op) model, dataset-4 held-out.

In [ ]:
# [22] Summary — consolidated headline numbers (all three passive models) + Fig 6 summary bars.
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
ax = axes[0]
groups = ["pos R²\n(MLP)", "vel R²\n(late MLP)", "1 - fiber resid\n(MLP g)"]
gv = {n: [pos_res[(n, "mlp")]["r2"], vel_res[(n, "late")][("sf", "mlp")]["r2"], 1 - fiber[n]["mlp_rf"]] for n in MODEL_ORDER}
x = np.arange(3); w = 0.8 / len(MODEL_ORDER)
for j, n in enumerate(MODEL_ORDER):
    off = (j - (len(MODEL_ORDER) - 1) / 2) * w
    ax.bar(x + off, gv[n], w * 0.92, color=MCOLOR[n], label=n)
    for i, v in enumerate(gv[n]):
        ax.text(x[i] + off, v + 0.012, f"{v:.2f}", ha="center", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=8.5); ax.set_ylim(0, 1.12)
ax.set_ylabel("score (0-1, higher better)"); ax.set_title("(a) recoverability & canonicality (passive)")
ax.legend(fontsize=8); style_ax(ax)

ax = axes[1]
ednames = ["Readout injection", "MLP-probe gradient", "Global-PCA projection", "PCA geodesic"]
def best_editor(n):
    return max(ednames, key=lambda e: METRICS[n][e]["pct_swap"])
bx = np.arange(len(MODEL_ORDER))
best_ps = [METRICS[n][best_editor(n)]["pct_swap"] for n in MODEL_ORDER]
ax.bar(bx, best_ps, 0.5, color=[MCOLOR[n] for n in MODEL_ORDER])
ax.axhline(100, color="0.3", ls="--", lw=1.2, label="true-state swap (100%)")
for i, n in enumerate(MODEL_ORDER):
    ax.text(bx[i], best_ps[i] + 1.5, f"{best_editor(n).split()[0]}\n{best_ps[i]:.0f}%", ha="center", fontsize=7)
ax.set_xticks(bx); ax.set_xticklabels([n.split()[0] for n in MODEL_ORDER], fontsize=8)
ax.set_ylabel("obs-change (% of true-state swap)"); ax.set_title("(b) §4 editability: best structural editor")
ax.legend(fontsize=7); style_ax(ax)
fig.suptitle("Fig 6 - Summary across the three passive models", y=1.03, fontsize=13)
fig.tight_layout(); fig.savefig(f"{OUT}/fig6_summary.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

display(Markdown("### Consolidated headline numbers (passive; **3->2 = action-knowledge**, 1->3 = perturbation-diversity)"))
display(Markdown("**§1-§3 geometry / recoverability / canonicality**"))
both_table({n: dict(twonn=GEO[n]["twonn"], curv=GEO[n]["curv"], pos_mlp=pos_res[(n, "mlp")]["r2"],
                    vel_mlp=vel_res[(n, "late")][("sf", "mlp")]["r2"], fib=fiber[n]["mlp_rf"]) for n in MODEL_ORDER},
    [("twonn", "intrinsic dim", "{:.2f}"), ("curv", "curvature (deg)", "{:.1f}"), ("pos_mlp", "pos R² (MLP)", "{:.3f}"),
     ("vel_mlp", "vel R² (MLP)", "{:.3f}"), ("fib", "fiber resid (MLP)", "{:.3f}")], row_hdr="model")
display(Markdown("**§4 editability** — best structural editor obs-change (% of swap) + readout RMSE (lower=better)"))
both_table({n: dict(best=best_editor(n).split()[0], bestps=METRICS[n][best_editor(n)]["pct_swap"],
                    inj_read=METRICS[n]["Readout injection"]["readout"], geo_read=METRICS[n]["PCA geodesic"]["readout"],
                    swap_read=METRICS[n]["True-state swap"]["readout"], loo=REAL_LOO[n]) for n in MODEL_ORDER},
    [("best", "best editor", "{}"), ("bestps", "best obs-change %swap", "{:.1f}"),
     ("inj_read", "readout-inj RMSE", "{:.2f}"), ("geo_read", "geodesic RMSE", "{:.2f}"),
     ("swap_read", "true-swap RMSE", "{:.2f}"), ("loo", "real-state loo", "{:.2f}")], row_hdr="model")
pngs = sorted(os.path.join(OUT, f) for f in os.listdir(OUT) if f.endswith(".png"))
display(Markdown("**PNG manifest:**\n\n" + "\n".join(f"- `{p}`" for p in pngs)))
print(f"{len(pngs)} PNGs in {OUT}")
